In [285]:
import pandas as pd
from pathlib import Path
import matplotlib as plt
import scipy.stats as stats
import numpy as np
import seaborn as sns

In [286]:
directory = Path("180")

files = sorted(file for file in directory.rglob("*") if file.is_file())
countFiles = 0

print("Files type: ")
for file in files:
    print(file.suffix)
    countFiles += 1

print("Number of files: ", countFiles)



Files type: 
.csv
.csv
.csv
Number of files:  3


We have 3, each has csv type

In [287]:
gateways = pd.read_csv('180/gateways.csv', sep=';', on_bad_lines='warn')
readings = pd.read_csv('180/readings.csv', sep = r"\t", on_bad_lines='warn', engine='python')
units = pd.read_csv('180/units.csv', on_bad_lines='warn')



In [288]:
gateways.shape, readings.shape, units.shape

((16000, 14), (15572, 13), (2000, 13))

In [289]:
gateways.head()

,depot_id,vendor,modelHardware,id_unit,technician,Lon,gatewayId,address_mac,installDate,ipv4,signal_dbm,lastSeen,Lat,firmware_version
0,52,Pinho Lopes Lda.,GWX-200,1739.0,Irina Leal,-6.34286,GW-00000,c0:f5:84:f8:08:41,2024/06/06,188.32.87.90,-65.2,"10/10/2024, 00:00:00",40.57291,v2.7.0
1,113,Christoph GmbH,GWX-100,103.0,Carminho Fonseca,-6.61695,GW-00001,30:db:d5:75:e1:e2,"12/10/2021, 00:00:00",132.117.107.236,-78.7,2024-04-26,39.47290,v1.2.5
2,112,Rädel Löwer AG,GWX-100,13.0,Dr. Marc Robinson,-7.48817,GW-00002,cd:c6:2c:7f:93:2e,2024-04-21,114.142.159.228,-74.9,04 Nov 2024,39.16074,v1.2.5
3,104,Roy and Sons,NODE-9,1344.0,Ryan Rios,-8.59568,GW-00003,d5:03:c8:7c:50:b4,"12/03/2022, 00:00:00",164.198.143.169,-68.3,25 Jan 2025,39.01984,v1.2.5
4,36,Christoph GmbH,GWX-100,1324.0,César da Neto,-9.18714,GW-00004,93:41:75:8f:f1:d5,15 May 2018,65.233.19.254,-52.2,2024-08-06,39.47786,v1.2.5


In [290]:
readings.head()

,id_unit,cycleTs,currentMotor,dpFilter,oilTemperature,reservoirPressure,pressureOil,rms_vibration,flowmeter,humidity_intake,noiseAmbient,dischargeDewpoint,Failure
0,U00193,"08/27/2023, 06:29:00",4.05968,18.69811,64.86904,8.81915,3.75608,2.42950,27.70488,71.41778,71.88633,15.83173,0
1,U00757,2023/11/13,2.36328,49.17312,51.49558,8.07426,3.30342,4.46937,26.57501,65.59907,69.74589,11.00633,1
2,U01997,2023-11-26,7.33857,11.09040,48.07975,9.78198,4.56246,6.87273,32.19223,79.35666,78.22245,6.20960,0
3,U01633,2023-02-22,5.69969,19.58462,73.09699,9.65227,4.35662,NaN,27.33163,62.12112,72.77351,5.18222,1
4,U00467,"01/23/2023, 03:45:00",3.92084,22.86953,64.31103,8.68221,3.62936,6.36960,29.46377,43.40507,73.24571,6.12806,0


In [291]:
units.head()

,codeFleet,depot_id,hours_service,model_code,line_code,manufacturer,overhaulLast,id_unit,year_commission,region_install,serial_number,ownerOperator,duty_class
0,FA,105,20874.1,XC-7,L6,"Rivers, Welch and Hood","05/06/2023, 00:00:00",1,2019,Christinehaven,AJ-0056-27354,Huhn KGaA,3
1,FA,37,6896.4,XC-7,L4,Walker Ltd,2024-01-15,2,2023,Zerbst,ED-7134-75911,Rädel Löwer AG,heavy
2,FE,36,8193.9,APU-12,L4,Schenk GmbH,2023/01/19,3,2022,Guimarães,AT-0639-19922,Macedo Faria e Filhos,1
3,FB,102,18235.4,APU-20,L6,Silva,2024-03-14,4,2019,Lake Amy,JW-3703-50512,Reinhardt,3
4,FB,11,11350.3,XC-7,L5,Huhn KGaA,2019-05-07,5,2021,Fiães,UC-0462-35464,Leal Freitas S/A,1


###  Relationships between files

The files are linked by `id_unit`:

    units ──1:N──► readings
      └────1:1──► gateways

- **units → readings (1:N):** each unit has several readings.
- **units → gateways (1:1):** each unit has one gateway; `depot_id` matches in both files.

**Problems:**
- `id_unit` has a different format in each file (`193`, `193.0`, `U00193`), so it must be normalized.
- 162 readings (IDs U90001–U90497) have no matching unit (orphan records).
- 14,000 of the 16,000 gateways are not linked to any unit.

In [292]:
np.sort(units["id_unit"]), np.sort(gateways["id_unit"]), np.sort(readings["id_unit"])

(array([   1,    2,    3, ..., 1998, 1999, 2000]),
 array([ 1.,  2.,  3., ..., nan, nan, nan]),
 array(['U00001', 'U00001', 'U00001', ..., 'U90494', 'U90497', 'U90497'],
       dtype=object))

In [293]:
gateways["id_unit"].head(), readings["id_unit"].head(), units["id_unit"].head()
#data is related with id_unit, this column in both 3 files, the data type of id_unit column in each file is different

(0    1739.0
 1     103.0
 2      13.0
 3    1344.0
 4    1324.0
 Name: id_unit, dtype: float64,
 0    U00193
 1    U00757
 2    U01997
 3    U01633
 4    U00467
 Name: id_unit, dtype: object,
 0    1
 1    2
 2    3
 3    4
 4    5
 Name: id_unit, dtype: int64)

In [294]:
gateways["id_unit"] = gateways["id_unit"].astype("Int64")
units["id_unit"] = units["id_unit"].astype(int)
readings["id_unit"] = readings["id_unit"].astype(str).str.lstrip('U').astype(int)
gateways["id_unit"], readings["id_unit"], units["id_unit"]

(0        1739
 1         103
 2          13
 3        1344
 4        1324
          ... 
 15995    <NA>
 15996    <NA>
 15997    <NA>
 15998    <NA>
 15999    <NA>
 Name: id_unit, Length: 16000, dtype: Int64,
 0         193
 1         757
 2        1997
 3        1633
 4         467
          ... 
 15567    1758
 15568    1552
 15569    1052
 15570    1791
 15571    1290
 Name: id_unit, Length: 15572, dtype: int64,
 0          1
 1          2
 2          3
 3          4
 4          5
         ... 
 1995    1996
 1996    1997
 1997    1998
 1998    1999
 1999    2000
 Name: id_unit, Length: 2000, dtype: int64)

So the data is normalized now and id_unit now is type of int in all
 datasets.

In [297]:
gateways.dtypes, readings.dtypes, units.dtypes

(depot_id              int64
 vendor               object
 modelHardware        object
 id_unit               Int64
 technician           object
 Lon                 float64
 gatewayId            object
 address_mac          object
 installDate          object
 ipv4                 object
 signal_dbm          float64
 lastSeen             object
 Lat                 float64
 firmware_version     object
 dtype: object,
 id_unit                int64
 cycleTs               object
 currentMotor         float64
 dpFilter             float64
 oilTemperature       float64
 reservoirPressure    float64
 pressureOil          float64
 rms_vibration        float64
 flowmeter            float64
 humidity_intake      float64
 noiseAmbient         float64
 dischargeDewpoint    float64
 Failure                int64
 dtype: object,
 codeFleet           object
 depot_id             int64
 hours_service      float64
 model_code          object
 line_code           object
 manufacturer        object
 ove